In [5]:
pip install openpyxl

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip available: 22.3 -> 26.1.1
[notice] To update, run: C:\Users\suki\AppData\Local\Programs\Python\Python311\python.exe -m pip install --upgrade pip


In [6]:
import pandas as pd
import re

df = pd.read_csv('../data/processed/reclamations_dataset_final.csv')

# Dictionnaire des mots-clés par catégorie (à ajuster selon vos données)
keywords = {
    'broken_product': ['cassé', 'brisé', 'endommagé', 'défectueux', 'fonctionne pas', 'pété'],
    'late_delivery': ['retard', 'livré tard', 'pas reçu', 'attente', 'délai', 'en retard'],
    'poor_quality': ['qualité', 'mauvais', 'médiocre', 'décevant', 'cher pour', 'non conforme'],
    'wrong_item': ['mauvais produit', 'erreur commande', 'pas le bon', 'reçu autre'],
    'transport_problem': ['transport', 'colis endommagé', 'livreur', 'emballage', 'perdu'],
    'admin_error': ['facturation', 'compte', 'remboursement', 'administratif', 'document'],
    'missing_item': ['manquant', 'il manque', 'incomplet', 'pièce manquante']
}

def predict_category(text):
    """Prédit la catégorie basée sur les mots-clés"""
    text = str(text).lower()
    scores = {}
    for cat, words in keywords.items():
        score = sum(1 for word in words if word in text)
        scores[cat] = score
    
    # Retourner la catégorie avec le score le plus élevé
    best_cat = max(scores, key=scores.get)
    # Si score maximum est 0, retourner 'autre'
    if scores[best_cat] == 0:
        return 'autre'
    return best_cat

# Prédire sur un échantillon
df_sample = df.sample(100, random_state=42)
df_sample['predicted'] = df_sample['text_fr'].apply(predict_category)

# Calculer la concordance (à défaut de vérité terrain)
# On regarde si les prédictions suivent le même pattern que les catégories existantes
from scipy.stats import chi2_contingency
contingency = pd.crosstab(df_sample['categorie'], df_sample['predicted'])
print("Matrice de confusion :")
print(contingency)

# Score : cohérence entre l'ancien et nouveau labeling
agreement = (df_sample['categorie'] == df_sample['predicted']).sum()
print(f"\nTaux de concordance : {agreement/len(df_sample)*100:.1f}%")

KeyError: 'text_fr'

In [4]:
# After you fill the Excel file, reload and calculate accuracy
validated = pd.read_excel('../data/processed/validation_sample.xlsx')

correct = (validated['categorie'] == validated['validated_category']).sum()
total = len(validated)
print(f"Auto-labeling accuracy: {correct/total*100:.1f}%")

Auto-labeling accuracy: 0.0%
